# Document analysis with SecBert trained classifier

---


Mount your own drive space as working space with the following three commands

In [ ]:
# !pip install -U spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def spacy_sent_tokenize(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

In [ ]:
# !pip install pandas
# !pip3 install torch torchvision
# !pip install transformers
# !pip install sklearn

In [ ]:
import torch
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from sklearn.preprocessing import LabelEncoder

from time import sleep

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import shutil

models = ["secbert"]

for model in models:
    zip_path = f"/content/drive/My Drive/mitre_model_{model}.zip"
    local_path = f"/content/mitre_model_{model}.zip"
    extract_path = f"/content/mitre_model_{model}"

    shutil.copy(zip_path, local_path)
    with zipfile.ZipFile(local_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("jackaduma/SecBERT")


In [ ]:
# Setting up the device for GPU usage

from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
device

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv("dataset_mixed.csv")

assert 'sentence' in df.columns and 'label_tec' in df.columns, "Missing required columns!"
df['sentence'] = df['sentence'].astype(str)

import joblib
encoder = joblib.load("mitre_model_secbert/label_encoder.pkl")

LABELS = 193

#Dataset Sanity
print(df['label_tec'].isna().sum())
df = df.dropna(subset=['label_tec'])  # removing rows with missing labels
valid_labels = set(encoder.classes_)
df = df[df['label_tec'].isin(valid_labels)]

df['enc_label'] = encoder.transform(df['label_tec'])
label_counts = df['label_tec'].value_counts()
df = df[df['label_tec'].isin(label_counts[label_counts >= 2].index)]
train_dataset, test_dataset = train_test_split(df, test_size=0.2, stratify=df['enc_label'], random_state=42)

# Reset index
train_dataset = train_dataset.reset_index(drop=True)
test_dataset = test_dataset.reset_index(drop=True)


In [ ]:
train_dataset = pd.read_csv('train_dataset_tram_split.csv')
test_dataset = pd.read_csv('test_dataset_tram_split.csv')
import joblib
encoder = joblib.load("mitre_model_secbert/label_encoder.pkl")

# Adding encoded labels
if 'enc_label' not in test_dataset.columns:
    test_dataset['enc_label'] = encoder.transform(test_dataset['label_tec'])

In [ ]:
# Defining some key variables/configurations
MAX_LEN = 512
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-05

In [ ]:
class Triage(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.len = len(dataframe)
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        sentence = str(self.data.sentence[index])
        sentence = " ".join(sentence.split())
        inputs = self.tokenizer.encode_plus(
            sentence,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']

        if 'enc_label' not in self.data:
            return {
            'sentence': sentence,
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long)
            }

        return {
            'sentence': sentence,
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'targets': torch.tensor(self.data.enc_label[index], dtype=torch.long)
        }

    def __len__(self):
        return self.len

In [ ]:
print("FULL Dataset: {}".format(df.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

training_set = Triage(train_dataset, tokenizer, MAX_LEN)
testing_set = Triage(test_dataset, tokenizer, MAX_LEN)

train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [ ]:
# Creating the customized SecBert model, by adding a drop out and a dense layer on top of distil bert to get the final output for the model.

class SecBertClass(torch.nn.Module):
    def __init__(self, pretrained_model_name: str, num_classes: int = None, dropout: float = 0.3):
        super().__init__()
        config = BertConfig.from_pretrained(pretrained_model_name, output_hidden_states=True)
        self.model = AutoModel.from_pretrained(pretrained_model_name, config=config) #picking only the main body of the model
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(dropout)
        self.classifier = torch.nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        output_1 = self.model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.ReLU()(pooler)
        pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        return output

Load previous trained model

In [ ]:
#LOAD
model = SecBertClass("jackaduma/SecBERT", LABELS)
model.load_state_dict(torch.load('mitre_model_secbert/model.pt', map_location=torch.device('cpu')))

In [ ]:
model.to(device)

In [ ]:
len(train_dataset)

In [ ]:
# Creating the loss function and optimizer
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

# Function to calcuate the accuracy of the model (not used)
def calcuate_accu(big_idx, targets):
    n_correct = (big_idx==targets).sum().item()
    return n_correct

In [ ]:
def check_accuracy(loader, model):

    num_correct = 0
    num_samples = 0

    sentences = []
    predicted = []
    targets = []
    predictions_arr = []
    model.eval()

    with torch.no_grad():
      for i, data in enumerate(loader, 0):
          x = data['ids'].to(device, dtype = torch.long)
          mask = data['mask'].to(device, dtype = torch.long)
          y = data['targets'].to(device, dtype = torch.long)

          scores = model(x, mask)
          _, predictions = scores.max(1)
          num_correct += (predictions == y).sum()
          num_samples += predictions.size(0)

          sentences += data['sentence']
          predicted += scores
          predictions_arr += predictions
          targets += y

      print(
          f"Got {num_correct} / {num_samples} with accuracy {float(num_correct)/float(num_samples)*100:.2f}"
      )

      return predicted, targets, sentences, predictions_arr

In [ ]:
predicted, targets, sentences, predictions = check_accuracy(testing_loader, model)

In [ ]:
targets

In [ ]:
print(sentences[0], encoder.inverse_transform([predictions[0].item()])[0], encoder.inverse_transform([targets[0].item()])[0])

In [ ]:
check_accuracy(testing_loader, model)

In [ ]:
!pip install torchmetrics

In [ ]:
import numpy

In [ ]:
import torch
from torchmetrics import F1Score, Precision, Recall, Accuracy
f1 = F1Score(task="multiclass", num_classes=LABELS, average="macro").to(device)
preds = torch.stack(predicted).to(device)
pred_classes = preds.argmax(dim=1)
tags = torch.tensor(targets).to(device)
f1(preds, tags)

In [ ]:
precision = Precision(task="multiclass", num_classes=LABELS, average="macro").to(device)
precision(preds,tags)

In [ ]:
recall = Recall(task="multiclass", num_classes=LABELS, average="macro").to(device)
recall(preds, tags)

In [ ]:
top_k = Accuracy(task="multiclass", num_classes=LABELS).to(device)
top_k(preds, tags)

In [ ]:
predicted

In [ ]:
print(f"F1 Score:   {f1(pred_classes, tags):.4f}")
print(f"Precision:  {precision(pred_classes, tags):.4f}")
print(f"Recall:     {recall(pred_classes, tags):.4f}")
print(f"Accuracy:   {top_k(pred_classes, tags):.4f}")

In [ ]:
from nltk.tokenize import sent_tokenize

def remove_empty_lines(text):
	lines = text.split("\n")
	non_empty_lines = [line for line in lines if line.strip() != ""]

	string_without_empty_lines = ""
	for line in non_empty_lines:
		if line != "\n":
			string_without_empty_lines += line + "\n"

	return string_without_empty_lines

def combine_text(list_of_text):
    combined_text = ' '.join(list_of_text)
    return combined_text


In [ ]:
!python -m nltk.downloader punkt

In [ ]:
import yaml
import re

def repl(matchobj):
    return ","+ matchobj.group(1) + ","

def load_regex(filename):
    regex_list = []
    with open(filename, 'r') as val:
        document = yaml.safe_load(val)
        regex_list = document
    return regex_list

def apply_regex_to_string(regex_list, string):
    new_string = string
    for rex in regex_list:
        reg = rex.get('regex').strip()
        raw_s = r'{}'.format(reg)
        if re.search(raw_s, string):
            new_string = re.sub(raw_s, rex.get('code') + " ", string)
            break
    return new_string


In [ ]:
fin6_tec_2 = ['T1134', 'T1059', 'T1562', 'T1036', 'T1588', 'T1003', 'T1021', 'T1569', 'T1078', 'T1102',
        'T1087', 'T1482', 'T1069', 'T1018', 'T1016', 'T1548', 'T1071', 'T1185', 'T1059',
        'T1543', 'T1132', 'T1005', 'T1001', 'T1140', 'T1573', 'T1068', 'T1083',
        'T1564', 'T1562', 'T1070', 'T1105', 'T1056', 'T1112', 'T1046', 'T1095', 'T1027', 'T1137',
        'T1003', 'T1069', 'T1057', 'T1055', 'T1572', 'T1572', 'T1090', 'T1012', 'T1620', 'T1021',
        'T1018', 'T1029', 'T1113', 'T1518', 'T1553', 'T1218', 'T1049', 'T1007', 'T1569', 'T1550',
        'T1078', 'T1047']

fin6_tec_1 = ['T1087', 'T1560', 'T1119', 'T1547', 'T1110', 'T1059', 'T1074', 'T1573', 'T1068', 'T1070',
        'T1046', 'T1003', 'T1572', 'T1021', 'T1018', 'T1053', 'T1078', 'T1003']

# MenuPass [8]

menuPass_tec_8 = ['T1560', 'T1119', 'T1059', 'T1005', 'T1074', 'T1210', 'T1083', 'T1574', 'T1106', 'T1027', 'T1003', 'T1199', 'T1078', 'T1047']

adFind = ['T1087', 'T1482', 'T1069', 'T1018', 'T1016']

certutil = ['T1140', 'T1105', 'T1553']

quasarRAT = ['T1059', 'T1555', 'T1573', 'T1105', 'T1056', 'T1112', 'T1090', 'T1021', 'T1053', 'T1553', 'T1082', 'T1552', 'T1125']

menuPass_tec_8.extend(adFind)
menuPass_tec_8.extend(certutil)
menuPass_tec_8.extend(quasarRAT)

# MenuPass [2]

menuPass_tec_2 = ['T1583', 'T1560', 'T1568', 'T1070', 'T1056', 'T1036', 'T1105', 'T1566', 'T1021', 'T1199', 'T1204', 'T1078']

poisonIvy = ['T1010', 'T1547', 'T1059', 'T1543', 'T1005', 'T1074', 'T1573', 'T1105', 'T1056', 'T1112', 'T1027',
'T1055', 'T1014']

menuPass_tec_2.extend(poisonIvy)

# WizardSpider [2]

wizardSpider_tec_2 = ['T1547', 'T1059', 'T1562', 'T1135', 'T1566', 'T1055', 'T1021', 'T1053', 'T1558', 'T1204', 'T1047']

bloodHound = ['T1087', 'T1560', 'T1059', 'T1482', 'T1615', 'T1106', 'T1201', 'T1069', 'T1018', 'T1033']

cobaltStrike = ['T1548', 'T1134', 'T1087', 'T1071', 'T1197', 'T1185', 'T1059', 'T1043', 'T1543', 'T1132', 'T1005', 'T1001', 'T1030', 'T1140', 'T1573', 'T1203', 'T1068', 'T1083', 'T1564', 'T1562', 'T1070', 'T1105', 'T1056', 'T1112', 'T1026', 'T1106', 'T1046', 'T1135', 'T1095', 'T1027', 'T1137', 'T1003', 'T1069', 'T1057', 'T1055', 'T1572', 'T1090', 'T1012', 'T1620', 'T1021', 'T1018', 'T1029', 'T1113', 'T1518', 'T1553', 'T1218', 'T1016', 'T1049', 'T1007', 'T1569', 'T1550', 'T1078', 'T1047']

empire =  ['T1548', 'T1134', 'T1087', 'T1557', 'T1071', 'T1560', 'T1547', 'T1217', 'T1115', 'T1059', 'T1043', 'T1136', 'T1543', 'T1555', 'T1484', 'T1482', 'T1114', 'T1573', 'T1546', 'T1068', 'T1083', 'T1574', 'T1210', 'T1615', 'T1567', 'T1070',  'T1056', 'T1105', 'T1056', 'T1106', 'T1046', 'T1135', 'T1040', 'T1027', 'T1003', 'T1057', 'T1055', 'T1021', 'T1053', 'T1113', 'T1518', 'T1558', 'T1082', 'T1016', 'T1049', 'T1569', 'T1127', 'T1552', 'T1550', 'T1125', 'T1102', 'T1047']

mimikatz = ['T1134', 'T1098', 'T1547', 'T1555', 'T1003', 'T1207', 'T1558', 'T1552', 'T1550']

ping = ['T1018']

ryuk = ['T1134', 'T1547', 'T1059', 'T1486', 'T1083', 'T1222', 'T1562', 'T1490', 'T0828', 'T1036', 'T1106', 'T1027', 'T1057', 'T1055', 'T1021', 'T1053', 'T1489', 'T1082', 'T1614', 'T1016', 'T1205', 'T1078']

trickBot = ['T1087', 'T1087', 'T1071', 'T1547', 'T1185', 'T1110', 'T1059', 'T1059', 'T1043', 'T1543', 'T1555', 'T1555', 'T1132', 'T1005', 'T1140', 'T1482', 'T1573', 'T1041', 'T1210', 'T1008', 'T1083', 'T1495', 'T1562', 'T1105', 'T1056', 'T1559', 'T1036', 'T1112', 'T1106', 'T1135', 'T1571', 'T1027', 'T1027', 'T1069', 'T1566', 'T1566', 'T1542', 'T1057', 'T1055', 'T1055', 'T1090', 'T1219', 'T1021', 'T1018', 'T1053', 'T1553', 'T1082', 'T1016', 'T1033', 'T1007', 'T1552', 'T1552', 'T1204', 'T1497']

wizardSpider_tec_2.extend(bloodHound)
wizardSpider_tec_2.extend(cobaltStrike)
wizardSpider_tec_2.extend(empire)
wizardSpider_tec_2.extend(mimikatz)
wizardSpider_tec_2.extend(ping)
wizardSpider_tec_2.extend(ryuk)
wizardSpider_tec_2.extend(trickBot)

#WizardSpider [7]

wizardSpider_tec_7 = ['T1087', 'T1059', 'T1048', 'T1210', 'T1562', 'T1027', 'T1021', 'T1018', 'T1489', 'T1518', 'T1558', 'T1082', 'T1569']

adFind = ['T1087', 'T1482', 't1069', 'T1018', 'T1016']

#CobaltStrike

net = ['T1087', 'T1087', 'T1136', 'T1136', 'T1070', 'T1135', 'T1201', 'T1069', 'T1069', 'T1021', 'T1018', 'T1049', 'T1007', 'T1569', 'T1124']

nltest = ['T1482', 'T1018', 'T1016']

#Ping

#Ryuk

wizardSpider_tec_7.extend(adFind)
wizardSpider_tec_7.extend(cobaltStrike)
wizardSpider_tec_7.extend(net)
wizardSpider_tec_7.extend(nltest)
wizardSpider_tec_7.extend(ping)
wizardSpider_tec_7.extend(ryuk)

In [ ]:
fin6_files = ['./Follow The Money-Dissecting the Operations of the Cyber Crime Group FIN6[1].txt',
                './Pick-Six-Intercepting a FIN6 Intrusion, an Actor Recently Tied to Ryuk and LockerGoga Ransomware[2].txt',
                './intelligence_summary.txt']

menuPass_files = ['./2018_12_20_united_states_v_zhu_hua_indictment[2].txt',
                './Japan-Linked Organizations Targeted in Long-Running and Sophisticated Attack Campaign[8].txt']

wizardSpider_files = ['./Ryuk’s Return[7].txt',
                     './Ransomware Activity Targeting the Healthcare and Public Health Sector. Retrieved October 28, 2020[2].txt']

In [ ]:
file_name = wizardSpider_files[1]
techniques = wizardSpider_tec_2

In [ ]:
# file_name = wizardSpider_files[0]
# techniques = wizardSpider_tec_7

In [ ]:
#Read report text from txt file
# import nltk
# nltk.download('punkt', force=True)  # Force fresh download

# from nltk.tokenize import sent_tokenize

lines = []
file_paths = [file_name]
for file_path in file_paths:
    with open(file_path) as f:
        lines += f.readlines()
import re
## Apply regex
regex_list = load_regex("regex.yml")

text = combine_text(lines)
text = re.sub('(%(\w+)%(\/[^\s]+))', repl, text)
text = apply_regex_to_string(regex_list, text)
text = re.sub('\(.*?\)', '', text)
text = remove_empty_lines(text)
text = text.strip()
sentences = spacy_sent_tokenize(text)
double_sentences = []

for i in range(1, len(sentences)):
    new_sen = sentences[i-1] + sentences[i]
    double_sentences.append(new_sen)

data = {'sentence': sentences}
df = pd.DataFrame(data, columns=['sentence'])
sentence_set = Triage(df, tokenizer, MAX_LEN)
testing_loader = DataLoader(sentence_set, **test_params)

In [ ]:
len(sentences)

In [ ]:
predicted = []
predict_proba_scores = []
with torch.no_grad():
      for i, data in enumerate(testing_loader, 0):
          x = data['ids'].to(device, dtype = torch.long)
          mask = data['mask'].to(device, dtype = torch.long)

          scores = model(x, mask)
          _, predictions = scores.max(1)
          proba_scores = torch.nn.functional.softmax(scores, dim=1)

          predicted += predictions
          predict_proba_scores += proba_scores


In [ ]:
predicted = [ pred.item() for pred in predicted]
predicted

In [ ]:
predict_proba_scores = [pred.max().item() for pred in predict_proba_scores]
predict_proba_scores

In [ ]:
print("Max predicted index:", max(predicted))
print("Encoder classes available:", len(encoder.classes_))


In [ ]:
predicted = encoder.inverse_transform(predicted)

In [ ]:
total = len(sentences)
correct = sum([1 for pred in predicted if pred in techniques])
accuracy = correct / total * 100
print(f"Raw Accuracy on CTI Report: {accuracy:.2f}%") # not true accuracy just an overlap check of known techniques and predicted techniques.

In [ ]:
def f_measure(recall, precision):
    if recall != 0 and precision != 0:
        return (2*precision*recall)/(precision+recall)
    else:
        return 0.01

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
precisions = []
recalls = []
corrected_pred = []
accepted_pred = []
correct_on_uniques = []
f1s = []

print(len(predicted))
print('list of predicted techniques: ',predicted)

lines = len(predicted)

In [ ]:
for threshold in thresholds:
    tecs = set(techniques)
    accepted = []

    for i in range(0,len(predict_proba_scores)):
        top_class = predicted[i]
        proba = predict_proba_scores[i]
        if proba > threshold:
            accepted.append(top_class)

    correct = 0

    unique_accepted = set(accepted)

    len_tecs = len(tecs)

    for pred in accepted:
        if pred in tecs: #True Positives
            correct += 1
    print('len of accepted: ',len(accepted))
    print('correct: ',correct)

    if len(accepted) != 0:
        precision = correct/len(accepted)*100
    else:
        precision = 0

    precision = round(precision,2)
    # print('precision: ',precision) #accuracy or precision?

    precisions.append(precision)

    for pred in accepted:
        if pred in tecs:
            tecs.remove(pred)

    recall = str(len_tecs-len(tecs))+ '/' + str(len_tecs)

    # print('recall: ',recall) #Recall

    recalls.append(recall)
    recall = (len_tecs-len(tecs))/len_tecs

    corrected_pred.append(correct)
    accepted_pred.append(len(accepted))

    cou = str(len_tecs-len(tecs))+ '/' + str(len(unique_accepted))
    correct_on_uniques.append(cou)
    cou = 0 if len(unique_accepted) == 0 else (len_tecs-len(tecs))/len(unique_accepted)

    # f1 = f_measure(recall=recall, precision=cou)
    f1 = f_measure(recall=recall, precision=precision / 100)
    f1 = round(f1,2)
    f1s.append(f1)

    print(f"Threshold {threshold:.2f}: F1 Score = {f1}, Precision = {round(precision,2)}, Recall = {round(recall,2)}")

In [ ]:
import matplotlib.pyplot as plt

# Converting string-form recall values to fractions
recalls_numeric = []
for r in recalls:
    if isinstance(r, str) and "/" in r:
        num, den = map(int, r.split("/"))
        recalls_numeric.append(num / den)
    else:
        recalls_numeric.append(r)

# Converting precision % to fraction (0-1 scale)
precisions_fraction = [p / 100 for p in precisions]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions_fraction, marker='o', label='Precision')
plt.plot(thresholds, recalls_numeric, marker='s', label='Recall')
plt.plot(thresholds, f1s, marker='^', label='F1 Score')

plt.xlabel('Confidence Threshold')
plt.ylabel('Score (0–1)')
plt.title('Precision, Recall, and F1 Score vs Threshold')
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
class Classifier_results:
    def __init__(self, title, lines, accepted_preds, correct_preds, precisions, recalls, correct_uniques, f1s):
        self.title = title
        self.lines = lines
        self.accepted_preds = accepted_preds
        self.correct_preds = correct_preds
        self.precisions = precisions
        self.recalls = recalls
        self.correct_uniques = correct_uniques
        self.f1s = f1s

class CSVOutput:
    def __init__(self, document_title, classifiers):
        self.classifiers = classifiers
        self.document_title = document_title

    def printify_array(self, array, sep = ';'):
        return sep + sep.join(str(x) for x in array)

    def _save_classifier_outputs(self, f):
        for classifier in self.classifiers:
            f.write(classifier.title + '\n')
            f.write(str(classifier.lines) + ' sentences\n')
            f.write('Accepted Predictions: {}\n'.format(self.printify_array(classifier.accepted_preds)))
            f.write('Corrected Predictions: {}\n'.format(self.printify_array(classifier.correct_preds)))
            f.write('Precision%: {}\n'.format(self.printify_array(classifier.precisions)))
            f.write('Recall%: {}\n'.format(self.printify_array(classifier.recalls)))
            f.write('Correct predictions on uniques: {}\n\n'.format(self.printify_array(classifier.correct_uniques)))

    def _save_classifier_f1(self, path):
        with open(path+'/'+self.document_title+'_f1.txt', 'w') as f:
            for classifier in self.classifiers:
                f.write(classifier.title)
                f.write(self.printify_array(classifier.f1s)+ '\n')

    def write_to_file(self, path):
        self._save_classifier_f1(path)
        with open(path+'/'+self.document_title+'.csv', 'w') as f:
            f.write('Tresholds; 0,1; 0.2; 0.3; 0.4; 0.5; 0.6; 0.7; 0.8;\n')
            self._save_classifier_outputs(f)

    def append_to_file(self, path):
        self._save_classifier_f1(path)
        with open(path+'/'+self.document_title+'.csv', 'a') as f:
            self._save_classifier_outputs(f)

In [ ]:
import pandas as pd


print("Sentence count:", len(sentences))
print("Predictions count:", len(predicted))
print("Confidence scores count:", len(predict_proba_scores))

assert len(sentences) == len(predicted) == len(predict_proba_scores), "Mismatch between predictions and sentences!"


output_df = pd.DataFrame({
    'Sentence': sentences,
    'Predicted Technique': predicted,
    'Confidence Score': predict_proba_scores
})

output_df.to_csv("secbert_cti_report_predictions.csv", index=False)
print("Predictions saved to cti_report_predictions.csv")

In [ ]:
result = Classifier_results( title='SecBert',
                              lines=lines,
                              accepted_preds=accepted_pred,
                              correct_preds=corrected_pred,
                              precisions=precisions,
                              recalls=recalls,
                              correct_uniques=correct_on_uniques,
                              f1s=f1s)

In [ ]:
fin6_ref_1_output = CSVOutput('secbert_cti_document_final_results', [result])
fin6_ref_1_output.write_to_file('.')